Libraries

In [1]:
import gradio as gr
import requests
import json
import logging
from SDS_functions import read_tsv_file
from gradio_callbacks import (
    analyze_and_get_additional, send_additional_wrapper, set_h_codes_dict, get_material_id_list, analyser_liste_ids, send_wrapper
)
import fitz
from io import BytesIO
from config import TSV_PATH, LOGIN_URL, UPLOAD_URL

# Loading of dictionary
H_CODES_DICT = read_tsv_file(TSV_PATH)
set_h_codes_dict(H_CODES_DICT)

# Loading of models
with open("models.txt", "r") as f:
    model_choices = [line.strip() for line in f if line.strip()]  # remove empty lines

# Clean existing handlers to avoid duplicates
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configuration of logging in both outputs (Jupyter + log file)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("notebook_run.log", encoding="utf-8"),
        logging.StreamHandler()  # In Jupyter cell
    ]
)


C:\Users\lombardi\AppData\Local\anaconda3\envs\CISPro_SDS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

demo = gr.Blocks()

with demo:
    gr.Markdown("# 🔬 SDS Parser")
    
    with gr.Accordion("Batch", open=False):
        with gr.Row():
            charger_btn = gr.Button("📥 Search for Material IDs of yesterday")
            analyser_tous_btn = gr.Button("⚙️ Analyze all")

        id_list_display = gr.Textbox(label="List of Material IDs", lines=5, placeholder="Z0001234\nZ0005678\nZ0009999")
        result_table = gr.Dataframe(label="Results of analysis", interactive=False)

        # Interactions
        charger_btn.click(fn=lambda: "\n".join(get_material_id_list()), outputs=id_list_display)
        analyser_tous_btn.click(fn=lambda mid: analyser_liste_ids(mid, H_CODES_DICT), inputs=id_list_display, outputs=result_table)
        
    with gr.Row():
        material_id_input = gr.Textbox(label="Material ID (Zxxxxx)", placeholder="Z0020315")
        analyse_btn = gr.Button("Analyze")

    with gr.Row():
        extract_ia_checkbox = gr.Checkbox(label="🔍 Use LLM to extract physical data", value=True)
        model_selector = gr.Dropdown(
            label="OpenRouter model",
            choices=model_choices,
            value=model_choices[0]
        )
    
    output_json = gr.Code(label="Result JSON", language="json")
    output_additional_json = gr.Code(label="Generated additional JSON", language="json")

    with gr.Row():
        username_input = gr.Textbox(label="Username")
        password_input = gr.Textbox(label="Password", type="password")
        send_btn = gr.Button("Send data to CISPro")
        send_additional_btn = gr.Button("Send additional data to CISPro")
    
    send_status = gr.Textbox(label="Status", interactive=False)

    analyse_btn.click(
        fn=analyze_and_get_additional,
        inputs=[material_id_input, extract_ia_checkbox, model_selector],
        outputs=[output_json, output_additional_json]
    )
    send_btn.click(fn=send_wrapper, inputs=[username_input, password_input, output_json], outputs=send_status)
    send_additional_btn.click(
        fn=send_additional_wrapper, 
        inputs=[username_input, password_input, output_additional_json], 
        outputs=send_status
    )

demo.launch()


2026-01-23 10:00:02,041 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7862


2026-01-23 10:00:02,196 [INFO] HTTP Request: GET http://127.0.0.1:7862/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-01-23 10:00:02,234 [INFO] HTTP Request: HEAD http://127.0.0.1:7862/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.
